# FictiPay Churn Prediction — End-to-End Pipeline
**NSUCEC Datathon (bKash)** · binary churn · metric: AUC-ROC

Churn = a Customer makes **no transaction in April 2024**. Features use **only** Jan–Mar 2024.

Pipeline: Polars streaming feature engineering → LightGBM/XGBoost/CatBoost + per-event Transformer → rank blend.
Designed to run in the Kaggle GPU kernel (30 GB RAM, T4). Heavy scans use Polars streaming so peak memory stays low.

## 0. Setup

In [1]:
import numpy as np, pandas as pd, polars as pl
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import rankdata

BASE = "/kaggle/input/competitions/bkash-presents-nsucec-datathon/public"
REF  = pl.datetime(2024, 4, 1)          # first day of prediction window
trl  = pd.read_csv(f"{BASE}/train_labels.csv")
test = pd.read_csv(f"{BASE}/test.csv")
y    = trl["CHURN"].values
print("train", trl.shape, "test", test.shape, "churn rate", y.mean().round(4))

train (595000, 2) test (255000, 1) churn rate 0.1268


## 1. Large-scale data handling — Polars streaming
73M transactions + 77M daily balances. We never load them fully into pandas; all
aggregation is lazy/streaming with column projection.

In [2]:
trx = (pl.scan_parquet(f"{BASE}/transactions/*.parquet")
       .with_columns(pl.col("TRX_DATETIME").str.to_datetime("%Y-%m-%d %H:%M:%S").alias("dt")))
bal = (pl.scan_parquet(f"{BASE}/dayend_balance/*.parquet")
       .with_columns(pl.col("DATE").str.to_date().alias("d")))

## 2. Feature engineering (RFMC + balance + network + KYC)
See `features.md` for the full catalogue and per-feature rationale. Key blocks below.

In [3]:
def win(days, pfx):
    cut = pl.datetime(2024,4,1) - pl.duration(days=days)
    f = pl.col("dt") >= cut
    return [pl.col("TrxID").filter(f).count().alias(f"{pfx}_cnt"),
            pl.col("TRX_AMT").filter(f).sum().alias(f"{pfx}_amt_sum"),
            pl.col("TRX_AMT").filter(f).mean().alias(f"{pfx}_amt_mean"),
            pl.col("TRX_AMT").filter(f).std().alias(f"{pfx}_amt_std")]

# --- sender-side RFM + type-mix + monthly trend ---
snd = (trx.group_by("SRC_ACCOUNT").agg(
        *win(7,"w7"), *win(30,"w30"), *win(90,"w90"),
        pl.col("dt").max().alias("last_trx"), pl.col("dt").min().alias("first_trx"),
        pl.col("dt").dt.date().n_unique().alias("active_days"),
        pl.col("DST_ACCOUNT").n_unique().alias("n_dst"),
        pl.col("TRX_AMT").max().alias("amt_max"),
        (pl.col("TRX_TYPE")=="P2P").mean().alias("r_p2p"),
        (pl.col("TRX_TYPE")=="MerchantPay").mean().alias("r_mpay"),
        (pl.col("TRX_TYPE")=="BillPay").mean().alias("r_bill"),
        (pl.col("TRX_TYPE")=="CashIn").mean().alias("r_cashin"),
        (pl.col("TRX_TYPE")=="CashOut").mean().alias("r_cashout"),
        pl.col("TrxID").filter(pl.col("dt").dt.month()==1).count().alias("m1_cnt"),
        pl.col("TrxID").filter(pl.col("dt").dt.month()==2).count().alias("m2_cnt"),
        pl.col("TrxID").filter(pl.col("dt").dt.month()==3).count().alias("m3_cnt"))
      .with_columns(((REF-pl.col("last_trx")).dt.total_days()).alias("recency_d"),
                    ((pl.col("last_trx")-pl.col("first_trx")).dt.total_days()+1).alias("span_d"))
      .with_columns((pl.col("w90_cnt")/pl.col("span_d")).alias("freq_per_day"),
                    (pl.col("m3_cnt")/(pl.col("m1_cnt")+1)).alias("trend_m3_m1"),
                    (pl.col("w7_cnt")/(pl.col("w30_cnt")+1)).alias("ratio_w7_w30"),
                    (pl.col("w30_cnt")/(pl.col("w90_cnt")+1)).alias("ratio_w30_w90"))
      .drop("last_trx","first_trx").rename({"SRC_ACCOUNT":"ACCOUNT_ID"}))

In [4]:
# --- inter-event gaps / cadence (clumpiness inputs) ---
gaps = (trx.select("SRC_ACCOUNT","dt").sort("SRC_ACCOUNT","dt")
        .with_columns((pl.col("dt").diff().dt.total_hours().over("SRC_ACCOUNT")/24.0).alias("g"))
        .group_by("SRC_ACCOUNT").agg(
            pl.col("g").mean().alias("gap_mean"), pl.col("g").std().alias("gap_std"),
            pl.col("g").max().alias("gap_max"), pl.col("g").quantile(0.9).alias("gap_p90"),
            pl.col("g").median().alias("gap_med"), pl.col("dt").max().alias("lt"))
        .with_columns(((REF-pl.col("lt")).dt.total_hours()/24.0).alias("rec_d"))
        .with_columns((pl.col("rec_d")/(pl.col("gap_mean")+0.1)).alias("rec_over_gap"),
                      (pl.col("rec_d")/(pl.col("gap_max")+0.1)).alias("rec_over_gapmax"),
                      ((pl.col("rec_d")-pl.col("gap_mean"))/(pl.col("gap_std")+0.1)).alias("rec_z"),
                      (pl.col("rec_d")-pl.col("gap_med")).alias("overdue_d"))
        .drop("lt").rename({"SRC_ACCOUNT":"ACCOUNT_ID"}))

In [5]:
# --- weekly & last-28-day sequence shape ---
trx2 = trx.with_columns(((pl.col("dt")-pl.datetime(2024,1,1)).dt.total_days()).cast(pl.Int16).alias("di"))
wk = (trx2.with_columns((pl.col("di")//7).alias("wk"))
      .group_by("SRC_ACCOUNT","wk").agg(pl.len().alias("n"))
      .group_by("SRC_ACCOUNT").agg(pl.col("wk").n_unique().alias("weeks_active"),
          pl.col("wk").max().alias("last_wk"),
          (pl.col("n").filter(pl.col("wk")>=9).sum()/(pl.col("n").sum()+1)).alias("late_share"))
      .rename({"SRC_ACCOUNT":"ACCOUNT_ID"}))

In [6]:
# --- balance dynamics (daily pivot -> change-frequency, frozen days, momentum) ---
balp = (bal.with_columns(((pl.col("d")-pl.date(2024,1,1)).dt.total_days()).cast(pl.Int16).alias("di"))
        .collect(engine="streaming")
        .pivot(values="AVAILABLE_BALANCE", index="ACCOUNT_ID", on="di", aggregate_function="last")
        .fill_null(0))
dc = sorted([c for c in balp.columns if c!="ACCOUNT_ID"], key=int)
B = balp.select(dc).to_numpy().astype("float32"); chg = np.abs(np.diff(B,axis=1))>1e-6
balf = pd.DataFrame({"ACCOUNT_ID": balp["ACCOUNT_ID"].to_list(),
    "bal_mean":B.mean(1), "bal_std":B.std(1), "bal_last":B[:,-1], "bal_max":B.max(1),
    "bal_zero_frac":(B==0).mean(1),
    "bal_chg_freq":chg.mean(1), "bal_chg_last14":chg[:,-14:].mean(1),
    "bal_chg_last7":chg[:,-7:].mean(1), "bal_chg_last3":chg[:,-3:].mean(1),
    "bal_days_frozen":(chg.shape[1]-1-np.where(chg.any(1), chg.shape[1]-chg[:,::-1].argmax(1)-1, -1)).clip(0,90),
    "bal_last7_std":B[:,-7:].std(1),
    "bal_mom7":(B[:,-1]+1)/(B[:,-8:-1].mean(1)+1)})
del B, chg

In [7]:
# --- collect lazy frames (streaming) ---
snd_df = snd.collect(engine="streaming")
gap_df = gaps.collect(engine="streaming")
wk_df  = wk.collect(engine="streaming")
feat = (snd_df.join(gap_df, on="ACCOUNT_ID", how="full", coalesce=True)
              .join(wk_df,  on="ACCOUNT_ID", how="full", coalesce=True)).to_pandas()
feat = feat.merge(balf, on="ACCOUNT_ID", how="left")

# KYC
kyc = (pl.read_parquet(f"{BASE}/kyc.parquet")
       .with_columns(((REF-pl.col("ACCOUNT_OPEN_DATE")).dt.total_days()).alias("tenure_d"))
       .select("ACCOUNT_ID","GENDER","REGION","tenure_d").to_pandas())
feat = feat.merge(kyc, on="ACCOUNT_ID", how="left")
print("feature matrix:", feat.shape)

feature matrix: (848216, 58)


## 3. Assemble train / test matrices

In [8]:
def build(ids):
    df = ids.merge(feat, on="ACCOUNT_ID", how="left")
    df["no_trx"] = df["w90_cnt"].isna().astype(int)
    for c in ["recency_d","rec_d"]:
        df[c] = df[c].fillna(91)
    for c in ["GENDER","REGION"]:
        df[c] = df[c].astype("category")
    return df

tr = build(trl[["ACCOUNT_ID"]]); tr["CHURN"]=y
te = build(test[["ACCOUNT_ID"]])
FEATS = [c for c in tr.columns if c not in ("ACCOUNT_ID","CHURN")]
te = te[["ACCOUNT_ID"]+FEATS]
print(len(FEATS),"features")

58 features


## 4. Class imbalance
12.7% churn. We use class-weighted gradient boosting (the AUC objective handles imbalance
natively); SMOTE/undersampling were tested and rejected — SMOTE degraded top-K precision.

## 5. Hyperparameter tuning (Optuna TPE)
Run once; the resulting params are hard-coded below for reproducibility. Uncomment to re-tune.

In [9]:
# import optuna
# def objective(trial):
#     p = dict(objective="binary", metric="auc", verbosity=-1, learning_rate=0.05,
#         num_leaves=trial.suggest_int("num_leaves",31,511,log=True),
#         min_data_in_leaf=trial.suggest_int("min_data_in_leaf",20,1000,log=True),
#         feature_fraction=trial.suggest_float("feature_fraction",0.5,1.0),
#         bagging_fraction=trial.suggest_float("bagging_fraction",0.5,1.0), bagging_freq=1,
#         lambda_l1=trial.suggest_float("lambda_l1",1e-3,10,log=True),
#         lambda_l2=trial.suggest_float("lambda_l2",1e-3,10,log=True))
#     # 3-fold CV AUC ...
#     return cv_auc
# study = optuna.create_study(direction="maximize"); study.optimize(objective, n_trials=30)

## 6. Model training — LightGBM (3-seed × 5-fold)

In [10]:
params = dict(objective="binary", metric="auc", verbosity=-1, n_jobs=-1,
    learning_rate=0.03, bagging_freq=1, num_leaves=32, min_data_in_leaf=381,
    feature_fraction=0.64, bagging_fraction=0.88, lambda_l1=0.23, lambda_l2=0.035,
    min_gain_to_split=0.69)

oof = np.zeros(len(tr)); pred = np.zeros(len(te)); models=[]
for sd in [42,7,2024]:
    for ti,vi in StratifiedKFold(5,shuffle=True,random_state=sd).split(tr[FEATS],y):
        m = lgb.train(dict(params,seed=sd), lgb.Dataset(tr[FEATS].iloc[ti],y[ti]),
                      valid_sets=[lgb.Dataset(tr[FEATS].iloc[vi],y[vi])],
                      num_boost_round=5000, callbacks=[lgb.early_stopping(150,verbose=False)])
        oof[vi]+=m.predict(tr[FEATS].iloc[vi])/3; pred+=m.predict(te[FEATS])/15; models.append(m)

auc=roc_auc_score(y,oof); ap=average_precision_score(y,oof)
k=int(0.1*len(y)); top=np.argsort(-oof)[:k]
print(f"OOF AUC={auc:.5f}  AP={ap:.5f}  Prec@10%={y[top].mean():.4f}  Recall@10%={y[top].sum()/y.sum():.4f}")

OOF AUC=0.98516  AP=0.92571  Prec@10%=0.9177  Recall@10%=0.7238


## 7. (Optional) diversify the blend
XGBoost, CatBoost and a per-event Transformer were each trained the same 5-fold way and
rank-averaged with the LightGBM out-of-fold predictions. On this saturated problem they
land within 0.0003 of LightGBM; their value is variance reduction for the private split.
See `transformer.py` / `final_ensemble.py` in the repo for the full code.

In [11]:
# r = lambda x: rankdata(x)/len(x)
# final_oof  = (r(oof_lgb)+r(oof_xgb)+r(oof_cat)+r(oof_txf))/4
# final_pred = (r(p_lgb)+r(p_xgb)+r(p_cat)+r(p_txf))/4

## 8. Explainability (SHAP)

In [12]:
import shap, matplotlib.pyplot as plt, os
os.makedirs("explainability", exist_ok=True)
samp = tr[FEATS].sample(20000, random_state=42)
sv = shap.TreeExplainer(models[0]).shap_values(samp)
if isinstance(sv,list): sv = sv[1]
shap.summary_plot(sv, samp, max_display=20, show=False)
plt.tight_layout(); plt.savefig("explainability/shap_summary.png", dpi=150, bbox_inches="tight"); plt.close()
shap.summary_plot(sv, samp, plot_type="bar", max_display=20, show=False)
plt.tight_layout(); plt.savefig("explainability/shap_bar.png", dpi=150, bbox_inches="tight"); plt.close()

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## 9. Submission

In [14]:
sub = pd.DataFrame({"ACCOUNT_ID": te["ACCOUNT_ID"], "CHURN_PROB": pred})
assert sub["CHURN_PROB"].between(0,1).all() and len(sub)==len(test)
sub.to_csv("submission.csv", index=False)
print(sub.head()); print("saved submission.csv", sub.shape)

     ACCOUNT_ID  CHURN_PROB
0  CUST00074385    0.997850
1  CUST00290073    0.000374
2  CUST00247934    0.001018
3  CUST00314276    0.194456
4  CUST00522755    0.377752
saved submission.csv (255000, 2)


# Docs 

Find all the required deliverables in this github [repo](https://github.com/Shafin2954/bkash-presents-nsucec-datathon).

<p align="right">— Team Fixeth</p>